# Reddit API Data - 2026 Refresh
This notebook now uses public Reddit JSON feeds instead of Pushshift and private Reddit credentials.
It refreshes the latest `r/wallstreetbets` posts, extracts ticker mentions, and saves fresh CSV files for the rest of the project.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    annotate_posts_with_tickers,
    build_mentions_table,
    ensure_nltk_resources,
    load_recent_posts,
)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
ensure_nltk_resources()
print(f"Project root: {ROOT}")

Project root: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
PER_FEED = 100
TOP_N = 10
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Fetch the latest public WallStreetBets posts

In [3]:
posts_df, source_name = load_recent_posts(PER_FEED)
posts_df = cast(pd.DataFrame, posts_df)
print(json.dumps({"data_source": source_name, "posts": int(len(posts_df))}, indent=2))
posts_df.head(10)

{
  "data_source": "reddit_public_json",
  "posts": 163
}


,id,feed,title,body,score,num_comments,created_utc,permalink,url,text
17,1t35lrn,hot,Best Trade EVER,"I have been trading for about 9 yrs as a part time thing and INTC has been incredible. \n\nAlmost 11,000% in under ...",1,1,2026-05-04 02:57:41+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t35lrn/best_trade_ever/,https://i.redd.it/hhsetppuc1zg1.jpeg,Best Trade EVER I have been trading for about 9 yrs as a part time thing and INTC has been incredible. \n\nAlmost 1...
4,1t34nnc,hot,This allowed here now? 😂,Major moves for a transformed company that wants to compete with Amazon 🤣 love it.,325,49,2026-05-04 02:13:37+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t34nnc/this_allowed_here_now/,https://i.redd.it/8x9f1ezy41zg1.jpeg,This allowed here now? 😂 Major moves for a transformed company that wants to compete with Amazon 🤣 love it.
10,1t312q5,hot,I don't even know why I sold..,"A win is a win.. But missed out on 25k? I jumped into SNDK, and holding on AMD leap for '27",33,32,2026-05-03 23:32:22+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t312q5/i_dont_even_know_why_i_sold/,https://i.redd.it/wqvd6tz7c0zg1.jpeg,"I don't even know why I sold.. A win is a win.. But missed out on 25k? I jumped into SNDK, and holding on AMD leap f..."
2,1t2xqe3,hot,Calls,,17930,229,2026-05-03 21:12:38+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2xqe3/calls/,https://i.redd.it/4jw18vjanzyg1.jpeg,Calls
9,1t2xamv,hot,Sold too soon,Thought I was doing good at 2500% gain.,69,52,2026-05-03 20:55:34+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2xamv/sold_too_soon/,https://i.redd.it/gp9wojx8kzyg1.jpeg,Sold too soon Thought I was doing good at 2500% gain.
1,1t2vr3z,hot,"What Are Your Moves Tomorrow, May 04, 2026",This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/w...,192,4590,2026-05-03 19:57:12+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2vr3z/what_are_your_moves_tomorrow_may_04_2026/,https://www.reddit.com/r/wallstreetbets/comments/1t2vr3z/what_are_your_moves_tomorrow_may_04_2026/,"What Are Your Moves Tomorrow, May 04, 2026 This post contains content not supported on old Reddit. [Click here to vi..."
7,1t2tuol,hot,SOUN YOLO,I am all IN \nSOUN 700 contracts 10.5 strike 05/08,327,141,2026-05-03 18:47:00+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2tuol/soun_yolo/,https://i.redd.it/zm2t046bxyyg1.jpeg,SOUN YOLO I am all IN \nSOUN 700 contracts 10.5 strike 05/08
3,1t2qu18,hot,Wise words Warren,,11140,244,2026-05-03 16:55:59+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2qu18/wise_words_warren/,https://i.redd.it/inkg6knhdyyg1.jpeg,Wise words Warren
5,1t2q994,hot,Every Layer of the AI Money Printer Got Front-Run. Except One.,Everyone called GPU bulls idiots in 2022. NVDA went from $150 to $950. Those idiots bought boats. \nThen memory. Tu...,1330,349,2026-05-03 16:34:17+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2q994/every_layer_of_the_ai_money_printer_got_frontrun/,https://www.reddit.com/r/wallstreetbets/comments/1t2q994/every_layer_of_the_ai_money_printer_got_frontrun/,Every Layer of the AI Money Printer Got Front-Run. Except One. Everyone called GPU bulls idiots in 2022. NVDA went f...
8,1t2p92k,hot,Best month Personal Record,Usually like the greats of this Reddit I’m usually in the Red. With the power of vibes and Intel I was able to beat ...,185,32,2026-05-03 15:56:21+00:00,https://www.reddit.com/r/wallstreetbets/comments/1t2p92k/best_month_personal_record/,https://i.redd.it/f6d3h4wu2yyg1.jpeg,Best month Personal Record Usually like the greats of this Reddit I’m usually in the Red. With the power of vibes an...


## Step 4 - Add sentiment and ticker extraction

In [4]:
annotated_posts_df = annotate_posts_with_tickers(posts_df)
annotated_posts_df = cast(pd.DataFrame, annotated_posts_df)
annotated_posts_df[["created_utc", "feed", "title", "score", "num_comments", "sentiment", "candidate_mentions"]].head(10)

,created_utc,feed,title,score,num_comments,sentiment,candidate_mentions
17,2026-05-04 02:57:41+00:00,hot,Best Trade EVER,1,1,0.7906,2
4,2026-05-04 02:13:37+00:00,hot,This allowed here now? 😂,325,49,0.7096,0
10,2026-05-03 23:32:22+00:00,hot,I don't even know why I sold..,33,32,-0.1027,2
2,2026-05-03 21:12:38+00:00,hot,Calls,17930,229,0.0000,0
9,2026-05-03 20:55:34+00:00,hot,Sold too soon,69,52,0.7430,0
1,2026-05-03 19:57:12+00:00,hot,"What Are Your Moves Tomorrow, May 04, 2026",192,4590,-0.2411,0
7,2026-05-03 18:47:00+00:00,hot,SOUN YOLO,327,141,0.3254,2
3,2026-05-03 16:55:59+00:00,hot,Wise words Warren,11140,244,0.4767,0
5,2026-05-03 16:34:17+00:00,hot,Every Layer of the AI Money Printer Got Front-Run. Except One.,1330,349,0.8997,5
8,2026-05-03 15:56:21+00:00,hot,Best month Personal Record,185,32,0.6597,0


## Step 5 - Rank the current top WallStreetBets tickers

In [5]:
summary_df, mentions_df = build_mentions_table(annotated_posts_df, TOP_N)
summary_df = cast(pd.DataFrame, summary_df)
mentions_df = cast(pd.DataFrame, mentions_df)
summary_df

,ticker,mention_count,post_count,avg_sentiment,total_score,total_comments,engagement,latest_mention,rank_score
31,POET,25,8,0.172825,6820,1471,86.842718,2026-05-01 18:15:12+00:00,57
20,INTC,14,8,0.412150,1351,451,60.025871,2026-05-04 02:57:41+00:00,46
26,NVDA,9,9,0.238189,4631,1123,94.912816,2026-05-03 16:34:17+00:00,45
0,AMD,13,5,0.089320,6147,928,46.735829,2026-05-03 23:32:22+00:00,33
37,SNDK,7,5,-0.007200,3097,919,46.226119,2026-05-03 23:32:22+00:00,27
46,VITL,14,3,0.771033,269,169,23.223145,2026-05-01 19:59:15+00:00,26
21,META,6,5,0.592940,1828,481,43.509286,2026-05-01 23:57:26+00:00,26
22,MSFT,5,5,-0.148480,7239,547,54.818770,2026-05-02 09:43:51+00:00,25
6,CAR,8,4,-0.046725,2794,767,37.335720,2026-05-02 08:33:33+00:00,24
44,TSLA,6,4,-0.072325,1198,486,40.490109,2026-05-02 09:43:51+00:00,22


## Step 6 - Save refreshed project CSV outputs
The notebook saves both the modern outputs under `outputs/latest_wsb_analysis/` and compatibility CSVs in the project root.

In [6]:
posts_export_df = annotated_posts_df.copy()
posts_export_df["timestamp"] = posts_export_df["created_utc"].dt.tz_convert(None)
posts_export_df["author"] = posts_export_df.get("author", "")
legacy_posts_df = posts_export_df[["title", "score", "body", "timestamp", "author"]].copy()
annotated_posts_df.to_csv(OUTPUT_DIR / "latest_wsb_posts.csv", index=False)
mentions_df.to_csv(OUTPUT_DIR / "latest_wsb_mentions.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "top10_wsb_stocks.csv", index=False)
legacy_posts_df.to_csv(ROOT / "wsb_reddit_api_data.csv", index=False)
legacy_posts_df.to_csv(ROOT / "wsb_pushshift_data.csv", index=False)
print("Saved files:")
for path in [
    OUTPUT_DIR / "latest_wsb_posts.csv",
    OUTPUT_DIR / "latest_wsb_mentions.csv",
    OUTPUT_DIR / "top10_wsb_stocks.csv",
    ROOT / "wsb_reddit_api_data.csv",
    ROOT / "wsb_pushshift_data.csv",
]:
    print(f"- {path}")

Saved files:
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\latest_wsb_posts.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\latest_wsb_mentions.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\top10_wsb_stocks.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\wsb_reddit_api_data.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\wsb_pushshift_data.csv


## Step 7 - Preview the refreshed legacy-compatible CSV format

In [7]:
legacy_posts_df.head(10)

,title,score,body,timestamp,author
17,Best Trade EVER,1,"I have been trading for about 9 yrs as a part time thing and INTC has been incredible. \n\nAlmost 11,000% in under ...",2026-05-04 02:57:41,
4,This allowed here now? 😂,325,Major moves for a transformed company that wants to compete with Amazon 🤣 love it.,2026-05-04 02:13:37,
10,I don't even know why I sold..,33,"A win is a win.. But missed out on 25k? I jumped into SNDK, and holding on AMD leap for '27",2026-05-03 23:32:22,
2,Calls,17930,,2026-05-03 21:12:38,
9,Sold too soon,69,Thought I was doing good at 2500% gain.,2026-05-03 20:55:34,
1,"What Are Your Moves Tomorrow, May 04, 2026",192,This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/w...,2026-05-03 19:57:12,
7,SOUN YOLO,327,I am all IN \nSOUN 700 contracts 10.5 strike 05/08,2026-05-03 18:47:00,
3,Wise words Warren,11140,,2026-05-03 16:55:59,
5,Every Layer of the AI Money Printer Got Front-Run. Except One.,1330,Everyone called GPU bulls idiots in 2022. NVDA went from $150 to $950. Those idiots bought boats. \nThen memory. Tu...,2026-05-03 16:34:17,
8,Best month Personal Record,185,Usually like the greats of this Reddit I’m usually in the Red. With the power of vibes and Intel I was able to beat ...,2026-05-03 15:56:21,


## Step 8 - Inspect the latest top 10 WSB stocks

In [8]:
summary_view = summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "engagement", "latest_mention"]].copy()
summary_view["latest_mention"] = pd.to_datetime(summary_view["latest_mention"], utc=True)
summary_view

,ticker,mention_count,post_count,avg_sentiment,engagement,latest_mention
31,POET,25,8,0.172825,86.842718,2026-05-01 18:15:12+00:00
20,INTC,14,8,0.412150,60.025871,2026-05-04 02:57:41+00:00
26,NVDA,9,9,0.238189,94.912816,2026-05-03 16:34:17+00:00
0,AMD,13,5,0.089320,46.735829,2026-05-03 23:32:22+00:00
37,SNDK,7,5,-0.007200,46.226119,2026-05-03 23:32:22+00:00
46,VITL,14,3,0.771033,23.223145,2026-05-01 19:59:15+00:00
21,META,6,5,0.592940,43.509286,2026-05-01 23:57:26+00:00
22,MSFT,5,5,-0.148480,54.818770,2026-05-02 09:43:51+00:00
6,CAR,8,4,-0.046725,37.335720,2026-05-02 08:33:33+00:00
44,TSLA,6,4,-0.072325,40.490109,2026-05-02 09:43:51+00:00


## Step 9 - Review sample mentions for the highest-ranked ticker

In [9]:
selected_ticker = str(summary_df.iloc[0]["ticker"])
selected_mentions_df = mentions_df.loc[mentions_df["ticker"].eq(selected_ticker)].copy()
selected_mentions_df["created_utc"] = pd.to_datetime(selected_mentions_df["created_utc"], utc=True)
print(f"Selected ticker: {selected_ticker}")
selected_mentions_df[["created_utc", "ticker", "mentions", "sentiment", "score", "num_comments", "title"]].head(20)

Selected ticker: POET


,created_utc,ticker,mentions,sentiment,score,num_comments,title
79,2026-05-01 18:15:12+00:00,POET,8,0.8962,5,14,"POET & I Know It, or POEL & I Go To Hell?"
194,2026-04-27 22:42:08+00:00,POET,4,0.9961,898,273,"POET YOLO “fail”, being the top signal, and learning from fellow Regards’ mistakes"
198,2026-04-27 20:38:52+00:00,POET,2,0.0000,941,240,$POET
199,2026-04-27 20:36:54+00:00,POET,1,-0.5423,225,48,POETry Loss
202,2026-04-27 17:56:27+00:00,POET,1,0.0000,1444,87,POET Puts: 6k --> 45k
203,2026-04-27 16:52:55+00:00,POET,1,-0.3182,1210,388,"POET loss, CFO should be in jail"
204,2026-04-27 16:19:03+00:00,POET,2,-0.3400,505,51,Dead $POET Society
211,2026-04-27 13:06:23+00:00,POET,6,0.6908,1592,370,POET -30% premarket after Marvell cancels Celestial AI purchase orders over alleged confidentiality breach


## Step 10 - Final notebook summary

In [10]:
notebook_summary = {
    "data_source": source_name,
    "posts_fetched": int(len(posts_df)),
    "latest_top_ticker": selected_ticker,
    "top_10_tickers": summary_df["ticker"].tolist(),
    "output_dir": str(OUTPUT_DIR.resolve()),
    "legacy_csvs": [
        str((ROOT / "wsb_reddit_api_data.csv").resolve()),
        str((ROOT / "wsb_pushshift_data.csv").resolve()),
    ],
}
print(json.dumps(notebook_summary, indent=2))

{
  "data_source": "reddit_public_json",
  "posts_fetched": 163,
  "latest_top_ticker": "POET",
  "top_10_tickers": [
    "POET",
    "INTC",
    "NVDA",
    "AMD",
    "SNDK",
    "VITL",
    "META",
    "MSFT",
    "CAR",
    "TSLA"
  ],
  "output_dir": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis",
  "legacy_csvs": [
    "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\wsb_reddit_api_data.csv",
    "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\wsb_pushshift_data.csv"
  ]
}
